# ReliableQA fine-tuning v3 — LoRA, bigger dataset, truncation-aware preprocessing

Changes from v2:
1. **LoRA (PEFT)** instead of full fine-tuning — base model frozen, only small adapters on `q_proj/k_proj/v_proj/o_proj` are trained.
2. **~10,000 training examples** (up from 2,500), ~65/35 answerable/unanswerable.
3. **Context truncation** instead of dropping long examples — question and answer are always preserved.
4. **Per-epoch task-accuracy validation** (not just LM loss) used to pick the best checkpoint.
5. **Extended metrics**: overall accuracy, answerable accuracy, unanswerable accuracy, exact match, answerable F1, and an overall SQuAD2-style F1.
6. Still a manual PyTorch training loop — no `Trainer`.


In [1]:
!pip install -q torch transformers datasets accelerate peft


## 1. Data — bigger, and no separate `prompt` string

We now store `context` and `question` separately (instead of a pre-built `prompt` string). This is required for step 3 below: we truncate the *context* at tokenize time to fit the token budget, so we can't bake the prompt text in ahead of time.

`n_answerable`/`n_unanswerable` are raised from 1,625/875 to 6,500/3,500 — a 4x increase, still close to the real SQuAD v2 ~65/35 ratio.

In [2]:
from datasets import load_dataset
import json, random

SEED = 42
random.seed(SEED)
dataset = load_dataset("rajpurkar/squad_v2")

def make_example(item):
    answers = item["answers"]["text"]
    answerable = len(answers) > 0
    return {
        "context": item["context"],
        "question": item["question"],
        "target": answers[0] if answerable else "UNANSWERABLE",
        "answerable": answerable,
        "all_answers": list(dict.fromkeys(answers)),
        "id": item["id"],
    }

train = dataset["train"].shuffle(seed=SEED)
train_ans = train.filter(lambda x: len(x["answers"]["text"]) > 0)
train_un = train.filter(lambda x: len(x["answers"]["text"]) == 0)

# ~10,000 training examples total, ~65/35 answerable/unanswerable (close to
# the real SQuAD v2 ratio). This is a 4x increase over v2's 2,500 examples,
# which was too little data for the model to reliably learn the
# answerable/unanswerable boundary.
n_answerable = 6500
n_unanswerable = 3500
train_data = [make_example(x) for x in train_ans.select(range(n_answerable))]
train_data += [make_example(x) for x in train_un.select(range(n_unanswerable))]
random.Random(SEED).shuffle(train_data)

validation = dataset["validation"].shuffle(seed=SEED)
val_data = [make_example(x) for x in validation.select(range(400))]
test_data = [make_example(x) for x in validation.select(range(400, 900))]

for name, data in [("train", train_data), ("validation", val_data), ("test", test_data)]:
    with open(f"{name}.jsonl", "w") as f:
        for item in data:
            f.write(json.dumps(item) + "\n")
    n_un = sum(not x["answerable"] for x in data)
    print(name, len(data), "unanswerable:", n_un)


README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

train 10000 unanswerable: 3500
validation 400 unanswerable: 209
test 500 unanswerable: 246


## 2. Model + LoRA setup

`get_peft_model` freezes every base-model weight automatically and inserts small trainable low-rank adapter matrices into the four attention projections. Only those adapters (a small fraction of the 360M parameters) will get gradients.

Note the learning rate: **2e-4** instead of v2's 2e-5. LoRA adapters start from (near-)zero and need a noticeably higher LR than full fine-tuning to actually learn something in a handful of epochs — 1e-4–3e-4 is the normal range for LoRA on a small model like this.

`EPOCHS` is lowered from 6 to 4: with 4x more data, each epoch already does ~4x more gradient updates, so fewer epochs are needed and it keeps Colab runtime reasonable.

In [3]:
import math
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModelForCausalLM, get_cosine_schedule_with_warmup
from peft import LoraConfig, get_peft_model, TaskType, PeftModel

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

MAX_LENGTH = 512
BATCH_SIZE = 4
ACCUM_STEPS = 4          # effective batch size = BATCH_SIZE * ACCUM_STEPS = 16
EPOCHS = 4
LEARNING_RATE = 2e-4     # higher than full-fine-tune LR: LoRA adapters start near zero
WEIGHT_DECAY = 0.01
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

INSTRUCTION = """You are a reliable QA assistant.
Use only the provided context.
If the answer is not supported by the context, respond exactly with UNANSWERABLE."""

def build_raw_prompt(context, question):
    """The requested prompt shape: instruction, then Context/Question/Answer."""
    return f"""{INSTRUCTION}

Context:
{context}

Question:
{question}

Answer:"""

def format_prompt(raw_prompt):
    """Wrap the instruction in the model\'s chat template. SmolLM2-Instruct
    was aligned on this format, so using it makes the UNANSWERABLE
    instruction much more reliable to follow than raw text."""
    messages = [{"role": "user", "content": raw_prompt}]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

# --- LoRA: freeze the base model, train small adapters on attention only ---
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(base_model, lora_config)
model.to(device)
model.print_trainable_parameters()   # expect well under 1% of parameters trainable


Device: cuda


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

## 3. Load the jsonl files back in (unchanged from v2)

In [ ]:
def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

train_data = read_jsonl("train.jsonl")
val_data = read_jsonl("validation.jsonl")

print("Train:", len(train_data))
print("Validation:", len(val_data))


## 4. Preprocessing — truncate the context, never drop the example

v2's `preprocess` returned `None` (silently dropping the example) whenever `prompt + target > MAX_LENGTH` tokens. That quietly shrank the training set and biased it toward shorter contexts.

`truncate_context_to_fit` instead:
1. Tokenizes the prompt with an *empty* context to measure how many tokens the instruction + question + chat-template wrapper + target already use ("overhead").
2. Subtracts that overhead (plus a small safety margin) from `MAX_LENGTH` to get a token budget for the context alone.
3. Truncates the context's token ids to that budget and decodes back to text.

The question and target are therefore always fully preserved — only the context gets cut, and only when it has to. A tiny hard-clip safety net at the end catches the rare case where re-tokenizing the decoded text drifts by a token or two.

In [ ]:
def truncate_context_to_fit(context, question, target_ids, tokenizer, max_length, safety_margin=8):
    overhead_prompt = format_prompt(build_raw_prompt("", question))
    overhead_len = len(tokenizer(overhead_prompt, add_special_tokens=False)["input_ids"])

    budget = max_length - overhead_len - len(target_ids) - safety_margin
    if budget < 20:
        budget = 20  # always leave a little context, even for unusually long questions

    context_ids = tokenizer(context, add_special_tokens=False)["input_ids"]
    if len(context_ids) <= budget:
        return context

    truncated_ids = context_ids[:budget]
    return tokenizer.decode(truncated_ids, skip_special_tokens=True)


def preprocess(example):
    target = example["target"] + tokenizer.eos_token
    target_ids = tokenizer(target, add_special_tokens=False)["input_ids"]

    truncated_context = truncate_context_to_fit(
        example["context"], example["question"], target_ids, tokenizer, MAX_LENGTH
    )
    raw_prompt = build_raw_prompt(truncated_context, example["question"])
    prompt = format_prompt(raw_prompt)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]

    input_ids = prompt_ids + target_ids

    # Safety net only — should rarely trigger given the budget above.
    if len(input_ids) > MAX_LENGTH:
        overflow = len(input_ids) - MAX_LENGTH
        prompt_ids = prompt_ids[:-overflow] if overflow < len(prompt_ids) else prompt_ids[:1]
        input_ids = prompt_ids + target_ids

    label = [-100] * len(prompt_ids) + target_ids
    return {"input_ids": input_ids, "labels": label}


train_example = [preprocess(example) for example in train_data]
val_example = [preprocess(example) for example in val_data]

print("Train:", len(train_example), "(none dropped)")
print("Validation:", len(val_example), "(none dropped)")


## 5. Generation helper (needed before training, for per-epoch accuracy checks)

In [ ]:
def generate_answer(model, context, question, tokenizer, device, max_new_tokens=32):
    # Reuse the same truncation logic as training, reserving room for the
    # generated tokens instead of a known target.
    truncated_context = truncate_context_to_fit(
        context, question, target_ids=[], tokenizer=tokenizer,
        max_length=MAX_LENGTH - max_new_tokens,
    )
    raw_prompt = build_raw_prompt(truncated_context, question)
    prompt = format_prompt(raw_prompt)
    input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"].to(device)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][input_ids.shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


## 6. Metrics (moved earlier — needed during training now, not just at the end)

`normalize_text`, `is_correct_prediction`, `token_f1`, `best_f1` are unchanged from v2. `squad2_f1` is new: a single F1 number computed over *every* example (1.0 for a correctly-abstained UNANSWERABLE, 0.0 if the answerable/unanswerable direction is wrong, token-F1 otherwise), so you get one F1 that summarizes the whole test set rather than only the answerable subset.

In [ ]:
import re
import string
from collections import Counter

def normalize_text(text):
    text = text.lower()
    text = "".join(ch for ch in text if ch not in string.punctuation)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())


def is_correct_prediction(example, prediction):
    prediction = normalize_text(prediction)
    if example["target"] == "UNANSWERABLE":
        return prediction == "unanswerable"
    accepted_answers = [normalize_text(answer) for answer in example["all_answers"]]
    return prediction in accepted_answers


def token_f1(prediction, ground_truth):
    """SQuAD-style token overlap F1. Gives partial credit to answers that
    are substantively right but not an exact string match."""
    pred_tokens = normalize_text(prediction).split()
    gt_tokens = normalize_text(ground_truth).split()

    if len(pred_tokens) == 0 or len(gt_tokens) == 0:
        return float(pred_tokens == gt_tokens)

    common = Counter(pred_tokens) & Counter(gt_tokens)
    num_same = sum(common.values())
    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


def best_f1(prediction, all_answers):
    return max((token_f1(prediction, ans) for ans in all_answers), default=0.0)


def squad2_f1(example, prediction):
    """SQuAD 2.0-style F1: 1.0 if both gold and prediction are UNANSWERABLE,
    0.0 if exactly one of them is, otherwise the normal token-overlap F1."""
    pred_is_unanswerable = normalize_text(prediction) == "unanswerable"
    gold_is_unanswerable = example["target"] == "UNANSWERABLE"

    if gold_is_unanswerable and pred_is_unanswerable:
        return 1.0
    if gold_is_unanswerable != pred_is_unanswerable:
        return 0.0
    return best_f1(prediction, example["all_answers"])


## 7. Dataset / DataLoader / collate (unchanged from v2)

In [ ]:
class QADataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __len__(self):
        return len(self.data)
    def __getitem__(self, idx):
        return self.data[idx]

train_dataset = QADataset(train_example)
val_dataset = QADataset(val_example)

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    input_ids, labels, attention_mask = [], [], []
    for x in batch:
        pad_len = max_len - len(x["input_ids"])
        ids = x["input_ids"] + [tokenizer.pad_token_id] * pad_len
        lab = x["labels"] + [-100] * pad_len
        att = [1] * len(x["input_ids"]) + [0] * pad_len
        input_ids.append(ids)
        labels.append(lab)
        attention_mask.append(att)
    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
    }

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)


## 8. Optimizer + scheduler

The only change: the optimizer is built from `trainable_params` (the LoRA adapters) instead of `model.parameters()`. Functionally similar since frozen params have no gradient anyway, but this is more memory-efficient and makes it explicit what's actually being trained.

In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(trainable_params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

steps_per_epoch = math.ceil(len(train_loader) / ACCUM_STEPS)
total_steps = steps_per_epoch * EPOCHS

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps,
)


## 9. Training step (gradient clipping kept, scoped to the trainable LoRA params)

In [ ]:
def train_epoch(model, train_loader, optimizer, scheduler, device, accum_steps=ACCUM_STEPS):
    model.train()
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids=input_ids, labels=labels, attention_mask=attention_mask)
        loss = outputs.loss / accum_steps
        loss.backward()
        total_loss += outputs.loss.item()

        is_last_batch = (step + 1) == len(train_loader)
        if (step + 1) % accum_steps == 0 or is_last_batch:
            torch.nn.utils.clip_grad_norm_(trainable_params, 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

    return total_loss / len(train_loader)


## 10. Validation — loss AND task accuracy

`evaluate_loss` is the same LM-loss pass as v2. `evaluate_accuracy` is new: it actually *generates* answers on a subset of validation examples (generation is much slower than a loss pass, so we use a subset per epoch rather than all 400) and reports overall/answerable/unanswerable accuracy. This is what decides the best checkpoint, since a lower loss doesn't guarantee better calibration on the answerable-vs-unanswerable decision.

In [ ]:
@torch.no_grad()
def evaluate_loss(model, val_loader, device):
    model.eval()
    total_loss = 0
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        labels = batch["labels"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        outputs = model(input_ids=input_ids, labels=labels, attention_mask=attention_mask)
        total_loss += outputs.loss.item()
    return total_loss / len(val_loader)


def evaluate_accuracy(model, data, tokenizer, device, max_examples=150):
    model.eval()
    subset = data[:max_examples]
    correct = 0
    answerable_correct = answerable_total = 0
    unanswerable_correct = unanswerable_total = 0

    for example in subset:
        answer = generate_answer(model, example["context"], example["question"], tokenizer, device)
        is_correct = is_correct_prediction(example, answer)
        if is_correct:
            correct += 1
        if example["answerable"]:
            answerable_total += 1
            answerable_correct += is_correct
        else:
            unanswerable_total += 1
            unanswerable_correct += is_correct

    return {
        "accuracy": correct / len(subset) if subset else 0.0,
        "answerable_accuracy": answerable_correct / answerable_total if answerable_total else 0.0,
        "unanswerable_accuracy": unanswerable_correct / unanswerable_total if unanswerable_total else 0.0,
    }


## 11. Training loop — best checkpoint chosen by validation *accuracy*, not loss

`model.save_pretrained` on a PEFT model only writes the small LoRA adapter (a few MB), not the full 360M-parameter model — much faster to save/load than v2's full checkpoints.

In [ ]:
SAVE_PATH = "./reliableqa-lora/final"
best_val_accuracy = -1.0

for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_loss = evaluate_loss(model, val_loader, device)
    val_metrics = evaluate_accuracy(model, val_data, tokenizer, device)

    acc = val_metrics["accuracy"]
    ans_acc = val_metrics["answerable_accuracy"]
    unans_acc = val_metrics["unanswerable_accuracy"]
    print(
        f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {acc:.3f} (answerable: {ans_acc:.3f}, unanswerable: {unans_acc:.3f})"
    )

    if val_metrics["accuracy"] > best_val_accuracy:
        best_val_accuracy = val_metrics["accuracy"]
        model.save_pretrained(SAVE_PATH)
        tokenizer.save_pretrained(SAVE_PATH)
        print(f"  -> new best (val accuracy {best_val_accuracy:.3f}), saved to {SAVE_PATH}")

print(f"\nBest validation accuracy: {best_val_accuracy:.3f}. Best adapter saved at {SAVE_PATH}")


## 12. Load the fine-tuned adapter for testing

A fresh base model + `PeftModel.from_pretrained` to attach the saved adapter (instead of `AutoModelForCausalLM.from_pretrained` on a full saved checkpoint).

In [ ]:
test_base_model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
test_model = PeftModel.from_pretrained(test_base_model, SAVE_PATH)
test_model.to(device)
test_model.eval()

test_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

print("Model loaded.")
print("Number of test examples:", len(test_data))


## 13. Quick sanity check on one example

In [ ]:
example = test_data[0]
answer = generate_answer(test_model, example["context"], example["question"], test_tokenizer, device)

print("Context:", example["context"][:300], "...")
print("\nQuestion:", example["question"])
print("\nExpected:", example["target"])
print("\nModel:", answer)


## 14. Full test-set evaluation with the requested metrics

Reports: overall accuracy, exact match (same value as accuracy here, since `is_correct_prediction` is already exact-match based), answerable accuracy, unanswerable accuracy, mean F1 over answerable examples, and the overall SQuAD2-style F1.

In [ ]:
def evaluate_test_set(model, test_data, tokenizer, device):
    total = len(test_data)
    correct = 0
    answerable_total = answerable_correct = 0
    unanswerable_total = unanswerable_correct = 0
    answerable_f1_sum = 0.0
    overall_f1_sum = 0.0

    results = []

    for i, example in enumerate(test_data):
        answer = generate_answer(model, example["context"], example["question"], tokenizer, device)
        expected = example["target"]
        is_correct = is_correct_prediction(example, answer)
        f1_overall = squad2_f1(example, answer)
        overall_f1_sum += f1_overall

        if is_correct:
            correct += 1

        if expected == "UNANSWERABLE":
            unanswerable_total += 1
            if is_correct:
                unanswerable_correct += 1
            f1 = None
        else:
            answerable_total += 1
            if is_correct:
                answerable_correct += 1
            f1 = best_f1(answer, example["all_answers"])
            answerable_f1_sum += f1

        results.append({
            "id": example["id"],
            "expected": expected,
            "model_answer": answer,
            "correct": is_correct,
            "answerable": expected != "UNANSWERABLE",
            "f1": f1,
            "f1_overall": f1_overall,
        })

        if (i + 1) % 20 == 0:
            print(f"Processed {i + 1}/{total}")

    overall_accuracy = correct / total if total else 0
    exact_match = overall_accuracy
    answerable_accuracy = answerable_correct / answerable_total if answerable_total else 0
    unanswerable_accuracy = unanswerable_correct / unanswerable_total if unanswerable_total else 0
    mean_answerable_f1 = answerable_f1_sum / answerable_total if answerable_total else 0
    mean_overall_f1 = overall_f1_sum / total if total else 0

    print(f"\nOverall accuracy:      {overall_accuracy:.3f}")
    print(f"Exact match:           {exact_match:.3f}")
    print(f"Answerable accuracy:   {answerable_accuracy:.3f}")
    print(f"Unanswerable accuracy: {unanswerable_accuracy:.3f}")
    print(f"Answerable mean F1:    {mean_answerable_f1:.3f}")
    print(f"Overall mean F1:       {mean_overall_f1:.3f}  (SQuAD2-style)")

    return results


results = evaluate_test_set(test_model, test_data, test_tokenizer, device)


## 15. Inspect wrong predictions (unchanged logic from v2)

In [ ]:
wrong = [r for r in results if not r["correct"]]
print(f"Wrong predictions: {len(wrong)}")

for r in wrong[:20]:
    print("\n--------------------")
    print("Expected:", r["expected"])
    print("Model:", r["model_answer"])
    print("Answerable:", r["answerable"])
    if r["f1"] is not None:
        print("F1 (partial credit):", round(r["f1"], 3))


## 16. Aggregate summary stats (unchanged logic from v2)

In [ ]:
unanswerable_predictions = sum(
    normalize_text(r["model_answer"]) == "unanswerable" for r in results
)

print("Total predictions:", len(results))
print("UNANSWERABLE predictions:", unanswerable_predictions)
print("UNANSWERABLE prediction rate:", unanswerable_predictions / len(results))

for label, value in [
    ("Correct", sum(r["correct"] for r in results)),
    ("Wrong", sum(not r["correct"] for r in results)),
    ("Model says UNANSWERABLE", unanswerable_predictions),
]:
    print(f"{label}: {value}")

answerable_wrong = [r for r in results if r["answerable"] and not r["correct"]]
unanswerable_wrong = [r for r in results if not r["answerable"] and not r["correct"]]

answerable_wrong_unanswerable = sum(
    normalize_text(r["model_answer"]) == "unanswerable" for r in answerable_wrong
)
unanswerable_wrong_real_answer = sum(
    normalize_text(r["model_answer"]) != "unanswerable" for r in unanswerable_wrong
)

print("Answerable wrong:", len(answerable_wrong))
print("Answerable -> incorrectly UNANSWERABLE:", answerable_wrong_unanswerable)
print("\nUnanswerable wrong:", len(unanswerable_wrong))
print("Unanswerable -> incorrectly answered:", unanswerable_wrong_real_answer)
